In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
df = pd.read_csv("loan_approval_dataset.csv")

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [3]:
df.isna().sum()

loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

In [4]:
df.drop("loan_id", axis=1, inplace=True)

In [5]:
encoder = LabelEncoder()

df["education"] = encoder.fit_transform(df["education"])

df["self_employed"] = encoder.fit_transform(df["self_employed"])

df["loan_status"] = encoder.fit_transform(df["loan_status"])

In [6]:
X = df.drop("loan_status", axis=1)

y = df["loan_status"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [9]:
model = Sequential()

model.add(Dense(32, activation="relu", input_shape=(X_train.shape[1],)))

model.add(Dense(16, activation="relu"))

model.add(Dense(8, activation="relu"))

model.add(Dense(1, activation="sigmoid"))

C:\Users\itsme\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [11]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6698 - loss: 0.6396 - val_accuracy: 0.8346 - val_loss: 0.5420
Epoch 2/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8785 - loss: 0.4055 - val_accuracy: 0.9034 - val_loss: 0.2814
Epoch 3/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9070 - loss: 0.2617 - val_accuracy: 0.9224 - val_loss: 0.2107
Epoch 4/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9206 - loss: 0.2209 - val_accuracy: 0.9253 - val_loss: 0.1864
Epoch 5/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9301 - loss: 0.2000 - val_accuracy: 0.9385 - val_loss: 0.1713
Epoch 6/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9341 - loss: 0.1812 - val_accuracy: 0.9429 - val_loss: 0.1555
Epoch 7/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9407 - loss: 0.1680 - val_accuracy: 0.9444 - val_loss: 0.1443
Epoch 8/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9444 - loss: 0.1561 - val_accuracy: 0.9473 - val_loss

In [12]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9379 - loss: 0.1254
Test Accuracy: 0.9379391074180603


In [13]:
y_pred = model.predict(X_test)

y_pred = (y_pred > 0.5).astype(int)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [14]:
print("Accuracy :", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")

print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")

print(classification_report(y_test, y_pred))

Accuracy : 0.9379391100702577

Confusion Matrix
[[509  27]
 [ 26 292]]

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       536
           1       0.92      0.92      0.92       318

    accuracy                           0.94       854
   macro avg       0.93      0.93      0.93       854
weighted avg       0.94      0.94      0.94       854



In [15]:
model.save("loan_ann_model.keras")

print("Model Saved Successfully")

Model Saved Successfully


In [16]:
def predict_loan():

    dependents = int(input("Dependents: "))

    education = input("Education (Graduate/Not Graduate): ")

    self_emp = input("Self Employed (Yes/No): ")

    income = float(input("Annual Income: "))

    loan_amount = float(input("Loan Amount: "))

    loan_term = int(input("Loan Term: "))

    cibil = int(input("CIBIL Score: "))

    residential = float(input("Residential Assets: "))

    commercial = float(input("Commercial Assets: "))

    luxury = float(input("Luxury Assets: "))

    bank = float(input("Bank Assets: "))

    education = 0 if education.lower() == "graduate" else 1

    self_emp = 1 if self_emp.lower() == "yes" else 0

    sample = [[
        dependents,
        education,
        self_emp,
        income,
        loan_amount,
        loan_term,
        cibil,
        residential,
        commercial,
        luxury,
        bank
    ]]

    sample = scaler.transform(sample)

    prediction = model.predict(sample)

    if prediction[0][0] > 0.5:
        print("Loan Approved")
    else:
        print("Loan Rejected")

predict_loan()

Dependents:  2
Education (Graduate/Not Graduate):  Graduate
Self Employed (Yes/No):  Yes
Annual Income:  23,00000


ValueError: could not convert string to float: '23,00000'